# Partial Cross-Entropy for Point-Supervised Remote Sensing Segmentation

This notebook demonstrates the full project end-to-end:

1. **Implement** the Partial Focal Cross-Entropy loss
2. **Simulate** sparse point annotations on real remote sensing data
3. **Train** a DeepLabV3+ model on those sparse labels (3-epoch mini demo)
4. **Explore** two factors: annotation density and focal weighting
5. **Extend** with SLIC label propagation and boundary-biased sampling

**Execution model:**
- Sections 1-4 run live (loss demo, data visualisation, 3-epoch mini-train ~5 min)
- Sections 5-8 load pre-computed results from `results/` -- no re-training

## Section 0 -- Setup

In [ ]:
import importlib, subprocess, sys
required = {
    'torch': 'torch',
    'segmentation_models_pytorch': 'segmentation-models-pytorch',
    'albumentations': 'albumentations',
    'skimage': 'scikit-image',
    'scipy': 'scipy',
    'tqdm': 'tqdm',
}
for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
print('All dependencies satisfied.')

In [ ]:
import os, csv, json, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from skimage.segmentation import slic as skimage_slic
from scipy.ndimage import distance_transform_edt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

warnings.filterwarnings('ignore')
%matplotlib inline

# ── Configure these two paths ─────────────────────────────────────────────────
DATA_ROOT   = '/home/ubuntu/weak-supervision/patches'
RESULTS_DIR = '/home/ubuntu/partial-ce-segmentation/results'
# ─────────────────────────────────────────────────────────────────────────────

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_imgs = len(os.listdir(os.path.join(DATA_ROOT, 'Images')))
print(f'Device  : {DEVICE}')
print(f'Images  : {n_imgs} found')
print(f'Results : {RESULTS_DIR}')

---
## Section 1 -- Partial Focal Cross-Entropy Loss

### What problem does this loss solve?

Standard cross-entropy requires a label for every pixel. We only have labels at a
handful of point locations. **Partial CE** masks the pixel-wise loss to those
locations -- unlabeled pixels contribute nothing to the gradient.

**Focal weighting** down-weights pixels the model already classifies confidently
(high p_t) and focuses attention on hard ones:

$$\mathcal{L}_{\text{pfCE}} = \frac{1}{|\mathcal{P}|} \sum_{i \in \mathcal{P}} -(1 - p_{t,i})^\gamma \cdot \log(p_{t,i} + \varepsilon)$$

Setting gamma=0 recovers plain partial CE.

In [ ]:
# Copied from losses/partial_ce.py
class PartialFocalCELoss(nn.Module):
    def __init__(self, gamma=2.0, num_classes=6):
        super().__init__()
        self.gamma = gamma
        self.num_classes = num_classes

    def forward(self, logits, targets, point_mask):
        '''
        logits:     [B, C, H, W] float32
        targets:    [B, H, W]    int64
        point_mask: [B, H, W]    float32 binary  (1=labeled, 0=ignore)
        '''
        total_points = point_mask.sum()
        if total_points == 0:
            return torch.tensor(0.0, requires_grad=True, device=logits.device)
        probs = F.softmax(logits, dim=1)
        targets_clamped = targets.clamp(0, self.num_classes - 1)
        p_t = probs.gather(1, targets_clamped.unsqueeze(1)).squeeze(1)
        focal_weight = (1.0 - p_t) ** self.gamma
        pixel_loss   = -focal_weight * torch.log(p_t + 1e-8)
        return (pixel_loss * point_mask).sum() / total_points.clamp(min=1)

In [ ]:
# Demo 1: basic forward pass
torch.manual_seed(0)
B, C, H, W = 2, 6, 32, 32
logits     = torch.randn(B, C, H, W)
targets    = torch.randint(0, C, (B, H, W))
point_mask = (torch.rand(B, H, W) < 0.01).float()
print(f'Labeled pixels: {int(point_mask.sum())} / {B*H*W}  ({100*point_mask.mean():.1f}%)')
criterion = PartialFocalCELoss(gamma=2.0)
print(f'Loss (gamma=2.0): {criterion(logits, targets, point_mask).item():.4f}')
print(f'Loss (empty mask): {criterion(logits, targets, torch.zeros(B,H,W)).item():.4f}  (expected 0.0)')

In [ ]:
# Demo 2: effect of gamma on loss value, and the focal weight curve
gammas = [0.0, 0.5, 1.0, 2.0]
print(f'  gamma   loss')
for g in gammas:
    l = PartialFocalCELoss(gamma=g)(logits, targets, point_mask).item()
    print(f'  {g:<5}  {l:.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
p = np.linspace(0.01, 0.99, 200)
colors_g = ['#999999', '#4caf50', '#2a7ae0', '#e05c2a']
for g, c in zip(gammas, colors_g):
    ax.plot(p, (1-p)**g, color=c, linewidth=2, label=f'gamma = {g}')
ax.set_xlabel('Model confidence p_t (probability for correct class)')
ax.set_ylabel('Focal weight (1 - p_t)^gamma')
ax.set_title('Focal weighting: contribution of each labeled pixel to the loss')
ax.legend(); ax.grid(alpha=0.3)
ax.annotate('Easy pixels heavily\ndown-weighted at gamma=2',
            xy=(0.85, 0.02), fontsize=9, color='#e05c2a',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
fig.tight_layout(); plt.show()

**Reading the chart:** at gamma=0 (grey) every labeled pixel gets weight 1.0 -- plain CE.
At gamma=2 (orange) a pixel the model classifies at 90% confidence gets weight
(1-0.9)^2 = 0.01 -- almost completely ignored.

**Why high gamma can hurt with sparse labels:** if you only have ~60 labeled pixels
per image and gamma=2 discards the 70% the model handles correctly, you may be left
learning from fewer than 20 pixels per image -- an extremely noisy signal.

---
## Section 2 -- Data and Point Simulation

**ISPRS Potsdam:** 2,400 aerial patches, 300x300 px, 6 semantic classes.
Instead of full pixel annotations, we simulate a human annotator clicking
a small number of points per class.

In [ ]:
# Copied from data/potsdam_dataset.py (lines 1-100)
COLOR_TO_CLASS = {
    (255,255,255): 0,  # Impervious surfaces
    (0,  0,  255): 1,  # Building
    (0,  255,255): 2,  # Low vegetation
    (0,  255,  0): 3,  # Tree
    (255,255,  0): 4,  # Car
    (255,  0,  0): 5,  # Clutter/background
}
CLASS_NAMES  = ['Impervious', 'Building', 'Low veg.', 'Tree', 'Car', 'Clutter']
CLASS_COLORS = np.array([[255,255,255],[0,0,255],[0,255,255],
                         [0,255,0],[255,255,0],[255,0,0]], dtype=np.uint8)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
SPLIT_RANGES  = {'train': (0, 1680), 'val': (1680, 2040), 'test': (2040, 2400)}

def rgb_to_label(label_rgb):
    H, W, _ = label_rgb.shape
    label = np.full((H, W), 5, dtype=np.int64)
    for rgb, cls in COLOR_TO_CLASS.items():
        match = ((label_rgb[:,:,0]==rgb[0]) & (label_rgb[:,:,1]==rgb[1])
                 & (label_rgb[:,:,2]==rgb[2]))
        label[match] = cls
    return label

def compute_boundary_distance(label):
    padded = np.pad(label, 1, mode='edge')
    is_boundary = (
        (padded[1:-1,1:-1] != padded[:-2, 1:-1]) |
        (padded[1:-1,1:-1] != padded[2:,  1:-1]) |
        (padded[1:-1,1:-1] != padded[1:-1,:-2])  |
        (padded[1:-1,1:-1] != padded[1:-1,2:]))
    return distance_transform_edt(~is_boundary).astype(np.float32)

def simulate_point_labels(label, points_per_class, sampling='uniform'):
    H, W = label.shape
    point_mask = np.zeros((H, W), dtype=np.float32)
    rng  = np.random.default_rng()
    dist = compute_boundary_distance(label) if sampling != 'uniform' else None
    for cls in range(6):
        coords = np.argwhere(label == cls)
        if len(coords) == 0: continue
        n = min(points_per_class, len(coords))
        if sampling == 'uniform':
            chosen = rng.choice(len(coords), size=n, replace=False)
        else:
            d = dist[coords[:,0], coords[:,1]].astype(np.float64)
            sigma = max(d.std(), 1.0)
            w = np.exp(-d/sigma) if sampling=='boundary' else np.exp(d/sigma)
            chosen = rng.choice(len(coords), size=n, replace=False, p=w/w.sum())
        point_mask[coords[chosen,0], coords[chosen,1]] = 1.0
    return point_mask

def propagate_points_to_superpixels(image, point_mask, n_segments=200, compactness=10.0):
    segments = skimage_slic(image.astype(np.float32)/255.0, n_segments=n_segments,
                            compactness=compactness, start_label=0, channel_axis=2)
    expanded = np.zeros_like(point_mask)
    for row, col in np.argwhere(point_mask > 0):
        expanded[segments == segments[row, col]] = 1.0
    return expanded

print('Helper functions defined.')

In [ ]:
# Load and display one example image
EXAMPLE_IDX = 100
img     = np.array(Image.open(f'{DATA_ROOT}/Images/Image_{EXAMPLE_IDX}.tif').convert('RGB'))
lbl_rgb = np.array(Image.open(f'{DATA_ROOT}/Labels/Label_{EXAMPLE_IDX}.tif').convert('RGB'))
label   = rgb_to_label(lbl_rgb)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img);                           axes[0].set_title('RGB image')
axes[1].imshow(CLASS_COLORS[label.clip(0,5)]); axes[1].set_title('Ground-truth label map')
for ax in axes: ax.axis('off')
patches = [mpatches.Patch(color=CLASS_COLORS[i]/255, label=CLASS_NAMES[i]) for i in range(6)]
fig.legend(handles=patches, loc='lower center', ncol=6, fontsize=9, bbox_to_anchor=(0.5,-0.04))
fig.tight_layout(); plt.show()

print('Pixel counts per class:')
for c, name in enumerate(CLASS_NAMES):
    n = (label==c).sum()
    print(f'  {name:<15}: {n:>6} px  ({100*n/label.size:.1f}%)')

In [ ]:
# Point density illustration: p = 1, 5, 10, 50
np.random.seed(42)
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
axes[0].imshow(img); axes[0].set_title('Image'); axes[0].axis('off')
for ax, ppc in zip(axes[1:], [1, 5, 10, 50]):
    mask = simulate_point_labels(label, ppc)
    ys, xs = np.where(mask > 0)
    dot_colors = [CLASS_COLORS[label[y,x]]/255.0 for y,x in zip(ys,xs)]
    ax.imshow(img, alpha=0.5)
    ax.scatter(xs, ys, c=dot_colors, s=14, linewidths=0.3, edgecolors='white')
    ax.set_title(f'p={ppc}  ({len(ys)} pts)'); ax.axis('off')
patches = [mpatches.Patch(color=CLASS_COLORS[i]/255, label=CLASS_NAMES[i]) for i in range(6)]
fig.legend(handles=patches, loc='lower center', ncol=6, fontsize=8, bbox_to_anchor=(0.5,-0.05))
fig.suptitle('Simulated point annotations at different densities', fontsize=12)
fig.tight_layout(); plt.show()

n_pts = sum(min(10, (label==c).sum()) for c in range(6))
print(f'At p=10: {n_pts} labeled pixels out of {label.size} total ({100*n_pts/label.size:.2f}% supervision)')

---
## Section 3 -- The Segmentation Model

**DeepLabV3+** with a ResNet-50 encoder: takes a full image, produces a
per-pixel class score map of the same spatial dimensions.
The partial CE loss then reads scores only at labeled point locations.

In [ ]:
# Copied from models/segmentation_model.py
def build_model(num_classes=6, backbone='resnet50', pretrained=True):
    encoder_weights = 'imagenet' if pretrained else None
    return smp.DeepLabV3Plus(encoder_name=backbone, encoder_weights=encoder_weights,
                             in_channels=3, classes=num_classes)

model = build_model(pretrained=False).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model      : DeepLabV3+ (ResNet-50 encoder)')
print(f'Parameters : {n_params:,}')

dummy = torch.randn(1, 3, 304, 304).to(DEVICE)
with torch.no_grad(): out = model(dummy)
print(f'Input  : {tuple(dummy.shape)}')
print(f'Output : {tuple(out.shape)}  (one score per class per pixel)')

---
## Section 4 -- Mini Training Demo

Train for **3 epochs on 100 images** to prove the pipeline works end-to-end.
This is not meant to produce a good model -- it demonstrates that partial CE
provides a real learning signal from sparse annotations.

The full experiments in Sections 5-8 used 50 epochs on 1,680 images.

In [ ]:
# Copied from data/potsdam_dataset.py (PotsdamPointDataset class)
class PotsdamPointDataset(Dataset):
    def __init__(self, data_root, split, points_per_class=10, augment=False,
                 use_slic=False, slic_n_segments=200, sampling='uniform', subset=None):
        self.data_root        = data_root
        self.points_per_class = points_per_class
        self.augment          = augment and (split == 'train')
        self.use_slic         = use_slic
        self.slic_n_segments  = slic_n_segments
        self.sampling         = sampling
        start, end = SPLIT_RANGES[split]
        self.indices = list(range(start, end))
        if subset is not None: self.indices = self.indices[:subset]
        self.aug = A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
                              A.RandomRotate90(p=0.5)]) if self.augment else None
        self.pad       = A.PadIfNeeded(min_height=304, min_width=304, border_mode=0)
        self.normalize = A.Compose([A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                                    ToTensorV2()])

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i     = self.indices[idx]
        image = np.array(Image.open(f'{self.data_root}/Images/Image_{i}.tif').convert('RGB'), dtype=np.uint8)
        label = rgb_to_label(np.array(Image.open(f'{self.data_root}/Labels/Label_{i}.tif').convert('RGB'), dtype=np.uint8))
        if self.aug:
            out   = self.aug(image=image, mask=label)
            image, label = out['image'], out['mask']
        point_mask = simulate_point_labels(label, self.points_per_class, self.sampling)
        if self.use_slic:
            point_mask = propagate_points_to_superpixels(image, point_mask, self.slic_n_segments)
        padded     = self.pad(image=image, masks=[label.astype(np.int32), (point_mask*255).astype(np.uint8)])
        image      = padded['image']
        label      = padded['masks'][0].astype(np.int64)
        point_mask = (padded['masks'][1] > 0).astype(np.float32)
        img_t = self.normalize(image=image)['image'].float()
        return {'image': img_t, 'label': torch.from_numpy(label).long(),
                'point_mask': torch.from_numpy(point_mask).float()}

print('PotsdamPointDataset defined.')

In [ ]:
# Copied from utils/metrics.py
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self._sum = 0.0; self._count = 0
    def update(self, val, n=1): self._sum += val*n; self._count += n
    @property
    def avg(self): return self._sum / self._count if self._count > 0 else 0.0

def compute_miou(preds, targets, num_classes=6):
    ious = []
    for cls in range(num_classes):
        t = targets == cls
        if not t.any(): continue
        p = preds == cls
        inter = (p & t).sum().item(); union = (p | t).sum().item()
        ious.append(inter / union if union > 0 else 0.0)
    return float(sum(ious)/len(ious)) if ious else 0.0

# Copied from train.py (train_one_epoch + validate)
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train(); meter = AverageMeter()
    for batch in tqdm(loader, desc='  train', leave=False):
        images, labels, pm = (batch['image'].to(device), batch['label'].to(device),
                               batch['point_mask'].to(device))
        optimizer.zero_grad()
        loss = criterion(model(images), labels, pm)
        loss.backward(); optimizer.step()
        meter.update(loss.item(), images.size(0))
    return meter.avg

@torch.no_grad()
def validate(model, loader, device, num_classes=6):
    model.eval(); all_preds, all_targets = [], []
    for batch in tqdm(loader, desc='  val', leave=False):
        all_preds.append(model(batch['image'].to(device)).argmax(1).cpu())
        all_targets.append(batch['label'])
    return compute_miou(torch.cat(all_preds), torch.cat(all_targets), num_classes)

print('Training utilities defined.')

In [ ]:
# Run the mini-train
MINI_EPOCHS = 3
MINI_SUBSET = 100
MINI_BATCH  = 8

mini_train_ds = PotsdamPointDataset(DATA_ROOT, 'train', points_per_class=10,
                                     augment=True,  subset=MINI_SUBSET)
mini_val_ds   = PotsdamPointDataset(DATA_ROOT, 'val',   points_per_class=10,
                                     augment=False, subset=40)
mini_train_loader = DataLoader(mini_train_ds, batch_size=MINI_BATCH,
                               shuffle=True,  num_workers=2, pin_memory=True)
mini_val_loader   = DataLoader(mini_val_ds,   batch_size=MINI_BATCH,
                               shuffle=False, num_workers=2, pin_memory=True)

mini_model     = build_model(pretrained=True).to(DEVICE)
mini_criterion = PartialFocalCELoss(gamma=2.0)
mini_optimizer = torch.optim.Adam(mini_model.parameters(), lr=1e-4, weight_decay=1e-4)
mini_history   = {'epoch': [], 'loss': [], 'val_miou': []}

print(f'Mini-training: {MINI_EPOCHS} epochs, {MINI_SUBSET} images, batch {MINI_BATCH}, device {DEVICE}')
for epoch in range(1, MINI_EPOCHS + 1):
    loss  = train_one_epoch(mini_model, mini_train_loader, mini_criterion, mini_optimizer, DEVICE)
    miou  = validate(mini_model, mini_val_loader, DEVICE)
    mini_history['epoch'].append(epoch)
    mini_history['loss'].append(loss)
    mini_history['val_miou'].append(miou)
    print(f'  Epoch {epoch}/{MINI_EPOCHS}  loss={loss:.4f}  val_mIoU={miou:.4f}')

In [ ]:
# Plot mini-train curves + qualitative inference
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(mini_history['epoch'], mini_history['loss'],
             'o-', color='#e05c2a', linewidth=2, markersize=8)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train loss')
axes[0].set_title('Training loss (mini-run)'); axes[0].grid(alpha=0.3)

axes[1].plot(mini_history['epoch'], mini_history['val_miou'],
             'o-', color='#2a7ae0', linewidth=2, markersize=8)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val mIoU')
axes[1].set_title('Validation mIoU (mini-run)'); axes[1].grid(alpha=0.3)

# Qualitative inference on the example image
mini_model.eval()
mean_t = torch.tensor(IMAGENET_MEAN).view(3,1,1)
std_t  = torch.tensor(IMAGENET_STD ).view(3,1,1)
img_t  = (torch.from_numpy(img).permute(2,0,1).float()/255.0 - mean_t) / std_t
img_t  = F.pad(img_t.unsqueeze(0), (0,4,0,4))
with torch.no_grad():
    pred = mini_model(img_t.to(DEVICE)).argmax(1)[0,:300,:300].cpu().numpy()
axes[2].imshow(CLASS_COLORS[pred.clip(0,5)]); axes[2].axis('off')
axes[2].set_title('Prediction after 3 epochs')

patches = [mpatches.Patch(color=CLASS_COLORS[i]/255, label=CLASS_NAMES[i]) for i in range(6)]
fig.legend(handles=patches, loc='lower center', ncol=6, fontsize=8, bbox_to_anchor=(0.5,-0.04))
fig.suptitle(f'3-epoch mini-train  ({MINI_SUBSET} images, p=10, gamma=2)', fontsize=12)
fig.tight_layout(); plt.show()

---
## Section 5 -- Experiment 1: Effect of Annotation Density

**Hypothesis:** more clicks per class -> higher mIoU, but with diminishing returns.
**Setup:** gamma fixed at 2.0. points_per_class swept over {1, 5, 10, 20, 50}.
**Results loaded from:** `results/summary.csv`

In [ ]:
def load_summary():
    with open(os.path.join(RESULTS_DIR, 'summary.csv')) as f:
        return [{k: (float(v) if k not in ('run_name','timestamp') else v)
                 for k,v in r.items()} for r in csv.DictReader(f)]

def load_history(run_name):
    path = os.path.join(RESULTS_DIR, 'runs', run_name, 'history.csv')
    epochs, losses, mious = [], [], []
    with open(path) as f:
        for r in csv.DictReader(f):
            epochs.append(int(r['epoch'])); losses.append(float(r['train_loss'])); mious.append(float(r['val_miou']))
    return epochs, losses, mious

summary = load_summary()
cls_keys = ['iou_impervious','iou_building','iou_low_vegetation','iou_tree','iou_car','iou_clutter']

exp1 = sorted([r for r in summary if r['run_name'].startswith('exp1_')],
              key=lambda r: r['points_per_class'])
print(f'  {"Run":<22} {"p":>4}  {"mIoU":>8}')
print('-'*38)
for r in exp1:
    print(f'  {r["run_name"]:<22} {int(r["points_per_class"]):>4}  {r["test_miou"]:>8.4f}')

In [ ]:
pts   = [int(r['points_per_class']) for r in exp1]
mious = [r['test_miou'] for r in exp1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(pts, mious, 'o-', color='#2a7ae0', linewidth=2, markersize=8)
for x, y in zip(pts, mious):
    ax1.annotate(f'{y:.3f}', (x,y), textcoords='offset points', xytext=(0,9), ha='center', fontsize=8)
ax1.set_xscale('log'); ax1.set_xticks(pts); ax1.set_xticklabels(pts)
ax1.set_xlabel('points_per_class'); ax1.set_ylabel('Test mIoU')
ax1.set_title('Overall mIoU vs annotation density'); ax1.grid(True, which='both', alpha=0.3)

colors_c = plt.cm.tab10(np.linspace(0, 0.6, 6))
for key, name, c in zip(cls_keys, CLASS_NAMES, colors_c):
    ax2.plot(pts, [r[key] for r in exp1], 'o-', color=c, linewidth=1.5, markersize=5, label=name)
ax2.set_xscale('log'); ax2.set_xticks(pts); ax2.set_xticklabels(pts)
ax2.set_xlabel('points_per_class'); ax2.set_ylabel('IoU')
ax2.set_title('Per-class IoU vs annotation density')
ax2.legend(fontsize=8, loc='lower right'); ax2.grid(True, which='both', alpha=0.3)
fig.suptitle('Experiment 1 -- Effect of annotation density  (gamma=2.0)', fontsize=12)
fig.tight_layout(); plt.show()

print('mIoU gains per density step:')
for i in range(len(pts)-1):
    print(f'  p={pts[i]}->{pts[i+1]}: {mious[i+1]-mious[i]:+.4f}')

**Key finding:** The largest jump is p=1->5. Returns diminish around p=10-20.
The resurgence at p=20->50 comes from dense annotation covering small objects
(cars, fragmented clutter) that uniform sampling at low density misses.

---
## Section 6 -- Experiment 2: Effect of Focal Weighting (gamma)

**Hypothesis:** moderate focal weighting helps; high gamma hurts when supervision is sparse.
**Setup:** p=10 fixed. gamma swept over {0.0, 0.5, 1.0, 2.0}.

In [ ]:
exp2 = sorted([r for r in summary if r['run_name'].startswith('exp2_')],
              key=lambda r: r['focal_gamma'])
gammas_e2 = [r['focal_gamma'] for r in exp2]
mious_e2  = [r['test_miou']   for r in exp2]

print(f'  {"Run":<22} {"gamma":>6}  {"mIoU":>8}')
print('-'*40)
for r in exp2:
    print(f'  {r["run_name"]:<22} {r["focal_gamma"]:>6.1f}  {r["test_miou"]:>8.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(gammas_e2, mious_e2, 's-', color='#e05c2a', linewidth=2, markersize=8)
for x,y in zip(gammas_e2, mious_e2):
    ax1.annotate(f'{y:.3f}', (x,y), textcoords='offset points', xytext=(0,9), ha='center', fontsize=8)
ax1.set_xticks(gammas_e2); ax1.set_xlabel('Focal gamma'); ax1.set_ylabel('Test mIoU')
ax1.set_title('Overall mIoU vs focal gamma'); ax1.grid(alpha=0.3); ax1.set_ylim(0.555, 0.600)

for key, name, c in zip(cls_keys, CLASS_NAMES, colors_c):
    ax2.plot(gammas_e2, [r[key] for r in exp2], 's-', color=c, linewidth=1.5, markersize=5, label=name)
ax2.set_xticks(gammas_e2); ax2.set_xlabel('Focal gamma'); ax2.set_ylabel('IoU')
ax2.set_title('Per-class IoU vs focal gamma')
ax2.legend(fontsize=8, loc='lower right'); ax2.grid(alpha=0.3)
fig.suptitle('Experiment 2 -- Effect of focal weighting  (p=10)', fontsize=12)
fig.tight_layout(); plt.show()

**Key finding:** gamma=0.5 or gamma=1 beats plain CE by ~1 pp. gamma=2 gains nothing
over gamma=1 because when supervision is already sparse, high gamma discards too many
of the few labeled pixels as 'easy', leaving the model with almost no learning signal.

---
## Section 7 -- SLIC Label Propagation

**Core idea:** one click = one supervised pixel. But the neighbours are probably
the same class. Expanding each click to its whole SLIC superpixel (~560 px)
multiplies supervised coverage 560x at zero extra annotation cost.

**Key question before training:** how pure are those superpixels?

In [ ]:
from skimage.segmentation import mark_boundaries

np.random.seed(42)
point_mask = simulate_point_labels(label, 10)
slic_mask  = propagate_points_to_superpixels(img, point_mask, n_segments=200)
segments   = skimage_slic(img.astype(np.float32)/255.0, n_segments=200,
                          compactness=10.0, start_label=0, channel_axis=2)

purity_map = np.full(label.shape, np.nan, dtype=np.float32)
for row, col in np.argwhere(point_mask > 0):
    sp = segments == segments[row, col]
    purity_map[sp] = float((label[sp] == int(label[row, col])).mean())

boundaries = mark_boundaries(img.astype(np.float32)/255.0, segments, color=(1,1,0), mode='thick')

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
axes[0].imshow(img);                           axes[0].set_title('Image')
axes[1].imshow(CLASS_COLORS[label.clip(0,5)]); axes[1].set_title('Ground truth')
axes[2].imshow(boundaries);                    axes[2].set_title('SLIC boundaries (n=200)')
im = axes[3].imshow(purity_map, cmap='RdYlGn', vmin=0, vmax=1)
ys, xs = np.where(point_mask > 0)
axes[3].scatter(xs, ys, c=[CLASS_COLORS[label[y,x]]/255.0 for y,x in zip(ys,xs)],
               s=18, linewidths=0.5, edgecolors='black')
axes[3].set_title('Superpixel purity map')
plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04, label='Purity')
for ax in axes: ax.axis('off')
fig.tight_layout(); plt.show()

valid_p = purity_map[~np.isnan(purity_map)]
print(f'Bare points      : {int(point_mask.sum())} px')
print(f'After SLIC expand: {int(slic_mask.sum())} px  ({int(slic_mask.sum())//int(point_mask.sum())}x expansion)')
print(f'Mean purity      : {valid_p.mean():.3f}')
print(f'Frac >= 0.80     : {(valid_p>=0.80).mean():.1%}')

In [ ]:
# Aggregate: purity vs n_segments sweep (load from CSV)
slic_records = []
with open(os.path.join(RESULTS_DIR, 'slic_analysis', 'purity_data.csv')) as f:
    for r in csv.DictReader(f):
        slic_records.append({'n_segments': int(r['n_segments']),
                              'purity': float(r['purity']), 'coverage': int(r['coverage'])})

n_segs = sorted(set(r['n_segments'] for r in slic_records))
mean_pur, frac_80, mean_cov = [], [], []
for n in n_segs:
    sub = [r for r in slic_records if r['n_segments']==n]
    pur = np.array([r['purity'] for r in sub]); cov = np.array([r['coverage'] for r in sub])
    mean_pur.append(pur.mean()); frac_80.append((pur>=0.80).mean()); mean_cov.append(cov.mean())

print(f'  {"n_seg":>8}  {"mean purity":>12}  {"frac>=0.80":>11}  {"mean coverage":>15}')
print('-'*55)
for n,p,f,c in zip(n_segs, mean_pur, frac_80, mean_cov):
    mark = '  <- recommended' if n==200 else ''
    print(f'  {n:>8}  {p:>12.4f}  {f:>10.1%}  {c:>15.1f}{mark}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(n_segs, mean_pur, 'o-', color='#2a7ae0', lw=2, ms=7, label='Mean purity')
ax1.plot(n_segs, frac_80,  's--', color='#e05c2a', lw=1.8, ms=6, label='Frac >= 0.80')
ax1.axvline(200, color='gray', ls=':', lw=1.5, label='n=200 (chosen)')
ax1.set(xlabel='n_segments', ylabel='Purity', ylim=(0,1.05), title='Purity vs granularity')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
ax2.plot(n_segs, mean_cov, 'o-', color='#4caf50', lw=2, ms=7)
ax2.axvline(200, color='gray', ls=':', lw=1.5)
ax2.set(xlabel='n_segments', ylabel='Mean px / superpixel', title='Coverage gain vs granularity')
ax2.grid(alpha=0.3)
fig.suptitle('SLIC granularity: more segments -> higher purity, lower coverage', fontsize=12)
fig.tight_layout(); plt.show()

In [ ]:
# Training result: SLIC vs point-only (load from disk)
runs_slic = [('exp1_p10_g2', 'Point-only  (p=10)', '#2a7ae0'),
             ('exp3_slic_p10_g2', 'SLIC-expanded (p=10)', '#e05c2a')]

print(f'  {"Run":<25} {"mIoU":>8}')
base_m = None
for rn, label_s, _ in runs_slic:
    m = json.load(open(os.path.join(RESULTS_DIR, 'runs', rn, 'test_metrics.json')))["test_miou"]
    if base_m is None: base_m = m
    print(f'  {rn:<25} {m:>8.4f}  ({m-base_m:+.4f})')
p50m = json.load(open(os.path.join(RESULTS_DIR, 'runs', 'exp1_p50_g2', 'test_metrics.json')))["test_miou"]
slic_m = json.load(open(os.path.join(RESULTS_DIR, 'runs', 'exp3_slic_p10_g2', 'test_metrics.json')))["test_miou"]
print(f'\nSLIC vs 5x more clicks (p=50): {slic_m - p50m:+.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
for rn, label_s, color in runs_slic:
    ep, _, mv = load_history(rn)
    ax.plot(ep, mv, lw=1.8, color=color, label=label_s)
    best = max(mv)
    ax.scatter([ep[mv.index(best)]], [best], color=color, s=60, zorder=5)
    ax.annotate(f'{best:.3f}', (ep[mv.index(best)], best),
                textcoords='offset points', xytext=(5,4), fontsize=8, color=color)
ax.set(xlabel='Epoch', ylabel='Val mIoU', title='Point-only vs SLIC-expanded supervision')
ax.legend(fontsize=9); ax.grid(alpha=0.3); fig.tight_layout(); plt.show()

**Key finding:** SLIC adds +3.7 pp mIoU -- more than collecting 5x as many clicks.
560 slightly-impure supervised pixels per click beats 1 perfectly-located pixel.
Coverage dominates over precision.

---
## Section 8 -- Boundary-Biased Sampling

**Question:** should annotators click near class boundaries or safely in the interior?

**The tradeoff:** boundary clicks carry more discriminative context but their local
neighbourhood is noisier. Interior clicks are very pure but informationally redundant --
the model can already infer what a large homogeneous region is.

In [ ]:
def local_purity(label, row, col, radius=5):
    H, W = label.shape
    r0,r1 = max(0,row-radius), min(H,row+radius+1)
    c0,c1 = max(0,col-radius), min(W,col+radius+1)
    return float((label[r0:r1,c0:c1] == label[row,col]).mean())

strat_labels = {'uniform': 'Uniform', 'boundary': 'Boundary-biased', 'interior': 'Interior-biased'}
strat_colors = {'uniform': '#2a7ae0', 'boundary': '#e05c2a', 'interior': '#4caf50'}
dist_map = compute_boundary_distance(label)

np.random.seed(42)
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
gt_blend = (img*0.55 + CLASS_COLORS[label.clip(0,5)]*0.45).astype(np.uint8)
axes[0].imshow(gt_blend); axes[0].set_title('Image + GT overlay'); axes[0].axis('off')

for ax, s in zip(axes[1:], ['uniform', 'boundary', 'interior']):
    mask = simulate_point_labels(label, 10, sampling=s)
    ys, xs = np.where(mask > 0)
    purities = np.array([local_purity(label, y, x) for y,x in zip(ys,xs)])
    ax.imshow(img, alpha=0.55)
    sc = ax.scatter(xs, ys, c=purities, cmap='RdYlGn', vmin=0, vmax=1,
                   s=25, linewidths=0.5, edgecolors='black', zorder=5)
    ax.contour(dist_map==0, levels=[0.5], colors=['white'], linewidths=[0.5], alpha=0.6)
    ax.set_title(f'{strat_labels[s]}\nmean purity={purities.mean():.3f}'); ax.axis('off')
plt.colorbar(sc, ax=axes[3], fraction=0.046, pad=0.04, label='Local purity')
fig.suptitle('Sampling strategies -- dots coloured by local purity (white=boundaries)', fontsize=12)
fig.tight_layout(); plt.show()

In [ ]:
# Aggregate diagnostic (load from CSV)
bnd_records = []
with open(os.path.join(RESULTS_DIR, 'boundary_analysis', 'sampling_data.csv')) as f:
    for r in csv.DictReader(f):
        bnd_records.append({'strategy': r['strategy'], 'class': int(r['class']),
                             'dist': float(r['dist_to_boundary']), 'purity': float(r['local_purity'])})

strategies = ['uniform', 'boundary', 'interior']
print(f'  {"Strategy":<18} {"N":>6}  {"mean dist":>10}  {"med dist":>9}  {"mean purity":>12}  {"frac>=0.90":>11}')
print('-'*75)
for s in strategies:
    sub = [r for r in bnd_records if r['strategy']==s]
    d = np.array([r['dist'] for r in sub]); p = np.array([r['purity'] for r in sub])
    print(f'  {strat_labels[s]:<18} {len(sub):>6}  {d.mean():>10.2f}  {np.median(d):>9.2f}  {p.mean():>12.4f}  {(p>=0.90).mean():>11.1%}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, metric, ylabel in [(ax1,'dist','Distance to boundary (px)'),
                            (ax2,'purity','Local purity (11x11 patch)')]:
    data  = [[r[metric] for r in bnd_records if r['strategy']==s] for s in strategies]
    parts = ax.violinplot(data, positions=[1,2,3], showmedians=True)
    for pc, s in zip(parts['bodies'], strategies):
        pc.set_facecolor(strat_colors[s]); pc.set_alpha(0.75)
    parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2)
    ax.set_xticks([1,2,3]); ax.set_xticklabels([strat_labels[s] for s in strategies])
    ax.set_ylabel(ylabel); ax.grid(axis='y', alpha=0.3)
fig.suptitle('Sampling strategy diagnostic: distance vs purity tradeoff', fontsize=12)
fig.tight_layout(); plt.show()

In [ ]:
# Training results: three-way comparison (load from disk)
runs_bnd = [('exp1_p10_g2',          'Uniform',         '#2a7ae0'),
            ('exp4_boundary_p10_g2', 'Boundary-biased', '#e05c2a'),
            ('exp4_interior_p10_g2', 'Interior-biased', '#4caf50')]

cls_keys_s = ['impervious','building','low_vegetation','tree','car','clutter']
metrics_bnd = {rn: json.load(open(os.path.join(RESULTS_DIR,'runs',rn,'test_metrics.json')))
               for rn,_,_ in runs_bnd}

# mIoU table
print(f'  {"Run":<28} {"Sampling":<18} {"mIoU":>8}  {"delta":>8}')
print('-'*70)
base_m = None
for rn, label_s, _ in runs_bnd:
    m = metrics_bnd[rn]['test_miou']
    if base_m is None: base_m = m
    print(f'  {rn:<28} {label_s:<18} {m:>8.4f}  {m-base_m:>+8.4f}')

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for rn, label_s, color in runs_bnd:
    ep, _, mv = load_history(rn)
    ax1.plot(ep, mv, lw=1.8, color=color, label=label_s)
    best = max(mv)
    ax1.scatter([ep[mv.index(best)]], [best], color=color, s=50, zorder=5)
    ax1.annotate(f'{best:.3f}', (ep[mv.index(best)],best),
                 textcoords='offset points', xytext=(5,4), fontsize=8, color=color)
ax1.set(xlabel='Epoch', ylabel='Val mIoU', title='Val mIoU: three sampling strategies')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

# Per-class bar chart
x = np.arange(6); width = 0.25
for i, (rn, label_s, color) in enumerate(runs_bnd):
    vals = [metrics_bnd[rn]['per_class_iou'][k] for k in cls_keys_s]
    ax2.bar(x+(i-1)*width, vals, width, label=label_s, color=color, alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(CLASS_NAMES, rotation=20, ha='right', fontsize=8)
ax2.set(ylabel='IoU', ylim=(0,1.0), title='Per-class test IoU')
ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)
fig.tight_layout(); plt.show()

**The purity paradox:** Interior-biased sampling has 97.4% mean local purity -- the
cleanest labels -- yet produces the worst model. Boundary-biased has 84.8% purity
yet beats uniform.

Clicks deep in a large homogeneous region confirm what the model could already infer.
Clicks near transitions are where the model needs signal most.
**Discriminative location beats label cleanliness.**

---
## Section 9 -- Summary and Conclusions

In [ ]:
# Full ranking table
ranked = sorted(summary, key=lambda r: r['test_miou'], reverse=True)
sup_map = {'exp3_slic_p10_g2': 'SLIC-expanded',
           'exp4_boundary_p10_g2': 'Boundary-biased',
           'exp4_interior_p10_g2': 'Interior-biased'}
print(f'  {"Rank":<6} {"Run":<28} {"Supervision":<18} {"p":>4}  {"gamma":>5}  {"mIoU":>8}')
print('-'*78)
for i, r in enumerate(ranked, 1):
    sup = sup_map.get(r['run_name'], 'Uniform')
    print(f'  {i:<6} {r["run_name"]:<28} {sup:<18} {int(r["points_per_class"]):>4}  {r["focal_gamma"]:>5.1f}  {r["test_miou"]:>8.4f}')

In [ ]:
# Key conditions bar chart
key_runs = [('exp1_p10_g2',          'Uniform\np=10',    '#aaaaaa'),
            ('exp1_p50_g2',          'Uniform\np=50',    '#2a7ae0'),
            ('exp3_slic_p10_g2',     'SLIC\np=10',       '#e05c2a'),
            ('exp4_boundary_p10_g2', 'Boundary\np=10',   '#9c27b0'),
            ('exp4_interior_p10_g2', 'Interior\np=10',   '#4caf50')]
by_name  = {r['run_name']: r for r in summary}
labels   = [l for _,l,_ in key_runs]
mious_k  = [by_name[rn]['test_miou'] for rn,_,_ in key_runs]
colors_k = [c for _,_,c in key_runs]
baseline = mious_k[0]

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(labels, mious_k, color=colors_k, edgecolor='white', linewidth=1.2, width=0.55)
ax.axhline(baseline, color='#aaaaaa', linestyle='--', lw=1.5,
           label=f'Uniform p=10 baseline ({baseline:.3f})')
for bar, v in zip(bars, mious_k):
    delta = v - baseline
    sign  = f'+{delta:.3f}' if delta >= 0 else f'{delta:.3f}'
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{v:.3f}\n({sign})', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
ax.set(ylabel='Test mIoU', ylim=(0.50, 0.65),
       title='Key conditions -- test mIoU (all p=10 except the p=50 bar)')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); plt.show()

## Conclusions

**Finding 1 -- Annotation density matters, but returns diminish.**
The largest jump is p=1->5. The plateau around p=10-20 is the practical sweet spot.
10 clicks per class is the annotation budget recommendation.

**Finding 2 -- Focal weighting: moderate helps, high hurts.**
gamma=0.5 or gamma=1 beats plain CE by ~1 pp. High gamma (2+) is no better because
it discards too many of the scarce labeled pixels as 'easy', starving the model of signal.

**Finding 3 -- Coverage dominates over precision (SLIC).**
Expanding each point to its SLIC superpixel adds +3.7 pp mIoU -- more than collecting
5x as many clicks (p=50 gains only +2.6 pp over p=10). Even with 26% of propagated
pixels being impure, the volume of coverage wins.

**Finding 4 -- The purity paradox (boundary sampling).**
Interior-biased sampling (97.4% purity) produces the worst model.
Boundary-biased sampling (84.8% purity) produces the best.
Discriminative location beats label cleanliness.

**The one-sentence takeaway:**
> In the sparse-supervision regime, *how much of the image each click supervises*
> matters more than *where exactly the click lands* -- coverage dominates.